# U Final Test: Frozen U1 MobileNetV2

**Project:** Explainable Transfer Learning Framework for Fetal Ultrasound Standard Plane Classification  
**Selected checkpoint:** `U1_MobileNetV2_frozen_seed42/best_model.keras`  
**Partition:** Original official **held-out TEST** (5,271 images; 896 patient groups)  
**Purpose:** One planned final evaluation after the U1 model-selection freeze dated 19 October 2026.

**Execution policy**

- **Run one numbered section at a time** and review its printed output before proceeding. **Do not use Run all.**
- Sections **1–5 do not decode, inspect, or evaluate test image pixels**. Section 4 checks ZIP *member metadata* (names/sizes) only. Section 5 requires a specific human authorization phrase.
- **Only Section 6 reads held-out test pixels**, after authorization and a one-time run lock. A saved partial prediction table is retained if Colab disconnects. **If Section 6 fails or is interrupted, STOP and report it; do not casually rerun Section 6.** Recovery must not modify the frozen model or tune on test feedback.
- **Section 7 computes predeclared metrics from stored predictions**, without calling the model or reading test image pixels again. Section 8 verifies exported evidence read-only.
- No training, no augmentation, no threshold optimization, no selection of a different checkpoint, no image resplitting, and no test-based tuning. The model outputs are uncalibrated softmax scores, not clinical probabilities.
- All outputs remain in the **ultrasound** project Drive folder. The original U1/U2 checkpoints, manifests, and Grad-CAM evidence are never overwritten.

## Planned metrics and split

On the unchanged 5,271-image official test partition, report accuracy, macro-precision, macro-recall, macro-F1, each class's precision/recall/F1/support, and the 6×6 confusion matrix. Keep the test results separate from the historical U1/U2 **validation** comparisons. There are no clinical, external-site, uncertainty-calibration, or model-superiority claims from this single test.

## Section 1: Verify the frozen U1 checkpoint and selection freeze (no test pixels)

Mount Drive when needed. Verify the actual selected checkpoint, its configuration, fixed patient-aware manifest, class mapping, and `model_selection_freeze` hashes. Refuse to proceed if a prior final-test output directory already exists. This section does **not** read test images.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
from zoneinfo import ZoneInfo
import hashlib
import json
import platform
import shutil
import zipfile

from google.colab import drive
import tensorflow as tf

ROOT = Path('/content/drive/MyDrive/ICTIU_Final_Projects')
if not ROOT.is_dir():
    drive.mount('/content/drive')
if not ROOT.is_dir():
    raise FileNotFoundError('Google Drive project root is unavailable.')

PROJECT = ROOT / 'ultrasound'
RUN_U1 = PROJECT / 'results/U1_MobileNetV2_frozen_seed42'
MODEL_PATH = RUN_U1 / 'best_model.keras'
CONFIG_PATH = RUN_U1 / 'configuration.json'
MANIFEST_PATH = PROJECT / 'splits/manifest_candidate.csv'
MAPPING_PATH = PROJECT / 'splits/class_mapping.json'
AUDIT_PATH = PROJECT / 'results/audit.json'
ZIP_PATH = PROJECT / 'archives/FETAL_PLANES_ZENODO.zip'
FREEZE_PATH = PROJECT / 'selection/U1_model_selection_freeze_2026-10-19.json'
HISTORICAL_VAL_PATH = RUN_U1 / 'validation_predictions.csv'
PROTOCOL_PATH = PROJECT / 'selection/U1_final_test_protocol_2026-10-19.json'
RESULT_DIR = PROJECT / 'results/U_Final_Test_U1_seed42'

files_required = [MODEL_PATH, CONFIG_PATH, MANIFEST_PATH, MAPPING_PATH,
                  AUDIT_PATH, ZIP_PATH, FREEZE_PATH, HISTORICAL_VAL_PATH]
missing = [str(p) for p in files_required if not p.is_file()]
if missing:
    raise FileNotFoundError('Required saved files missing: ' + str(missing))
TEST_RUN_STATE = 'NOT_STARTED'
if RESULT_DIR.exists():
    if (RESULT_DIR / 'FINALIZED.json').is_file():
        TEST_RUN_STATE = 'FINALIZED'
    elif (RESULT_DIR / 'test_inference_completed.json').is_file():
        TEST_RUN_STATE = 'INFERENCE_COMPLETED'
    else:
        raise RuntimeError(
            'An incomplete final TEST attempt is present. Stop and seek controlled recovery: '
            + str(RESULT_DIR)
        )
if PROTOCOL_PATH.exists():
    print('Existing final TEST protocol detected; Section 4 will verify and reuse it read-only.')

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as fh:
        for piece in iter(lambda: fh.read(4 * 1024 * 1024), b''):
            h.update(piece)
    return h.hexdigest()

freeze = json.loads(FREEZE_PATH.read_text(encoding='utf-8'))
assert freeze['decision_status'] == 'FROZEN_BEFORE_HELD_OUT_TEST'
assert freeze['selected_experiment'] == 'U1_MobileNetV2_frozen_seed42'
assert freeze['expected_test_images'] == 5271
assert freeze['final_test_protocol']['planned_evaluations'] == 1
assert freeze['final_test_protocol']['status_at_freeze'] == 'NOT_EVALUATED'
assert freeze['final_test_protocol']['test_based_tuning_allowed'] is False

for key, path in [('model_artifact', MODEL_PATH), ('configuration', CONFIG_PATH),
                  ('manifest', MANIFEST_PATH), ('mapping_file', MAPPING_PATH)]:
    claim = freeze[key]
    assert ROOT / claim['path_relative_to_root'] == path, key
    if sha256_file(path) != claim['sha256']:
        raise RuntimeError('Frozen source changed: ' + key)

KNOWN_U1_SHA = '5154c6d97685c2ef39d3ba040a409443cc0f261e070ca627c24b940979b897ae'
KNOWN_MANIFEST_SHA = 'ef6fec1a70e80a0b7ed34c990ebac2918ddc8a81dcef300aa244ce6ab1a5e1e0'
assert sha256_file(MODEL_PATH) == KNOWN_U1_SHA
assert sha256_file(MANIFEST_PATH) == KNOWN_MANIFEST_SHA

print('Python:', platform.python_version())
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
print('Frozen U1 model SHA-256:', sha256_file(MODEL_PATH))
print('Freeze record:', FREEZE_PATH.name)
print('Previous final TEST run state:', TEST_RUN_STATE)
print('SECTION 1 PASS: Frozen artifacts match. No held-out test pixels read.')

Mounted at /content/drive
Python: 3.13.15
TensorFlow: 2.21.0
GPU devices: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Frozen U1 model SHA-256: 5154c6d97685c2ef39d3ba040a409443cc0f261e070ca627c24b940979b897ae
Freeze record: U1_model_selection_freeze_2026-10-19.json
Previous final TEST run state: NOT_STARTED
SECTION 1 PASS: Frozen artifacts match. No held-out test pixels read.


## Section 2: Audit the official held-out split from metadata (no test pixels)

Read only the original 12,400-row manifest and audit files. Verify the class IDs, patient-group separation, split counts, original official-test assignment and unique image paths. Fix the **manifest row order** as evaluation order; do not shuffle or change any patient assignments. The test rows are metadata only at this point.

In [2]:
import pandas as pd

EXPECTED_CLASSES = {
    'Fetal abdomen': 0,
    'Fetal brain': 1,
    'Fetal femur': 2,
    'Fetal thorax': 3,
    'Maternal cervix': 4,
    'Other': 5,
}
class_to_id = json.loads(MAPPING_PATH.read_text(encoding='utf-8'))
config = json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
audit = json.loads(AUDIT_PATH.read_text(encoding='utf-8'))
assert class_to_id == EXPECTED_CLASSES == freeze['class_to_id']
assert audit['status'] == 'PASS_candidate_split_checks'
assert audit['manifest_sha256'] == sha256_file(MANIFEST_PATH)
assert audit['split_seed'] == 42
assert config['model'] == 'MobileNetV2'
assert config['backbone_frozen'] is True
assert config['test_used_for_model_selection'] is False
assert config['manifest_sha256'] == sha256_file(MANIFEST_PATH)

manifest = pd.read_csv(MANIFEST_PATH, dtype={'relative_path': str, 'stem': str,
                                             'group_id': str, 'sha256': str})
required_columns = {'relative_path', 'group_id', 'class_name', 'class_id',
                    'official_train', 'split', 'sha256'}
assert required_columns.issubset(manifest.columns), sorted(required_columns - set(manifest.columns))
assert len(manifest) == 12400
assert manifest['split'].value_counts().to_dict() == {
    'train': 5818, 'validation': 1311, 'test': 5271
}
assert manifest['relative_path'].is_unique
assert not manifest[list(required_columns)].isna().any().any()
assert (manifest['class_id'] == manifest['class_name'].map(class_to_id)).all()
assert set(manifest['class_id']) == set(range(6))
assert (manifest.groupby('group_id')['split'].nunique() == 1).all()
assert ((manifest['official_train'] == 0) == (manifest['split'] == 'test')).all()
assert manifest['sha256'].str.fullmatch(r'[0-9a-f]{64}').all()
assert (manifest.groupby('sha256')['split'].nunique() == 1).all()

counts = manifest.groupby('split')['group_id'].nunique().to_dict()
assert counts == {'test': 896, 'train': 716, 'validation': 180}

# Preserve exactly the saved manifest's test row order; NEVER re-split or shuffle.
test_rows = manifest.loc[manifest['split'] == 'test'].copy().reset_index(drop=True)
val_rows = manifest.loc[manifest['split'] == 'validation'].copy()
assert len(test_rows) == 5271
assert test_rows['group_id'].nunique() == 896
assert test_rows['class_id'].value_counts().min() > 0
CLASS_NAMES = [name for name, index in sorted(class_to_id.items(), key=lambda pair: pair[1])]

# Hash only existing metadata to seal the order before test-image access.
TEST_ORDER_SHA = hashlib.sha256(
    '\n'.join(test_rows['relative_path'].tolist()).encode('utf-8')
).hexdigest()
print('Manifest rows:', len(manifest))
print('Original official TEST images:', len(test_rows))
print('TEST patient groups:', test_rows['group_id'].nunique())
print('TEST class counts (metadata only):')
print(test_rows['class_name'].value_counts().reindex(CLASS_NAMES).to_string())
print('Fixed test row-order SHA-256:', TEST_ORDER_SHA)
print('SECTION 2 PASS: Original official held-out patient split verified; no test pixels read.')

Manifest rows: 12400
Original official TEST images: 5271
TEST patient groups: 896
TEST class counts (metadata only):
class_name
Fetal abdomen       358
Fetal brain        1472
Fetal femur         524
Fetal thorax        660
Maternal cervix     645
Other              1612
Fixed test row-order SHA-256: da93cb45f9cab66a7abc498fc116f7200253d4067b3a2c59a23fdffc170060f8
SECTION 2 PASS: Original official held-out patient split verified; no test pixels read.


## Section 3: Reproduce U1 inference on three validation images (no test pixels)

Load the saved model with `compile=False` and validate its exact internal Rescaling, frozen MobileNetV2 backbone, pooling/dropout and six-class head. Read **three validation images only** from the existing archive. Use the historical U1 validation prediction table to confirm that `tf.image.resize_with_pad` and built-in model scaling reproduce the original inference. No test pixels are opened.

In [3]:
import numpy as np
import stat
from pathlib import PurePosixPath
from tensorflow import keras

IMG_SIZE = 224
BATCH_SIZE = 16
saved_model = keras.models.load_model(str(MODEL_PATH), compile=False)
saved_model.trainable = False
rescaling = [l for l in saved_model.layers if isinstance(l, keras.layers.Rescaling)]
backbones = [l for l in saved_model.layers if isinstance(l, keras.Model)]
pools = [l for l in saved_model.layers if isinstance(l, keras.layers.GlobalAveragePooling2D)]
dropouts = [l for l in saved_model.layers if isinstance(l, keras.layers.Dropout)]
assert len(rescaling) == len(backbones) == len(pools) == len(dropouts) == 1
assert abs(float(rescaling[0].scale) - 1 / 127.5) < 1e-8
assert abs(float(rescaling[0].offset) + 1.0) < 1e-8
assert backbones[0].trainable is False
assert backbones[0].name.startswith('mobilenetv2')
head = saved_model.get_layer('class_scores')
assert isinstance(head, keras.layers.Dense) and head.units == 6
assert saved_model.input_shape == (None, 224, 224, 3)
assert saved_model.output_shape == (None, 6)

def validate_zip_member(name, zip_obj):
    rel = PurePosixPath(str(name))
    if rel.is_absolute() or '..' in rel.parts or '\\' in str(name):
        raise RuntimeError('Unsafe ZIP path: ' + str(name))
    if rel.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
        raise RuntimeError('Unexpected image extension: ' + str(name))
    member = zip_obj.getinfo(str(name))
    if (member.is_dir() or member.file_size <= 0 or
        member.file_size > 30 * 1024**2 or
        stat.S_ISLNK(member.external_attr >> 16)):
        raise RuntimeError('Unsafe or unexpected ZIP member: ' + str(name))
    return member

def decode_like_u1(image_bytes):
    decoded = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    decoded.set_shape((None, None, 3))
    resized = tf.image.resize_with_pad(decoded, IMG_SIZE, IMG_SIZE)
    return tf.cast(resized, tf.float32)  # Original model rescales internally.

# Predeclared validation-only smoke sample: first path in each of classes 0, 1, 2.
val_smoke = (val_rows.sort_values('relative_path')
             .groupby('class_id', sort=True).head(1)
             .sort_values('class_id').head(3).reset_index(drop=True))
assert val_smoke['class_id'].tolist() == [0, 1, 2]
validation_images = []
with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    for row in val_smoke.itertuples(index=False):
        validate_zip_member(row.relative_path, archive)
        raw = archive.read(row.relative_path)
        if hashlib.sha256(raw).hexdigest() != row.sha256:
            raise RuntimeError('Validation image hash mismatch: ' + row.relative_path)
        validation_images.append(decode_like_u1(raw))
validation_batch = tf.stack(validation_images)
assert tuple(validation_batch.shape) == (3, 224, 224, 3)
assert 0 <= float(tf.reduce_min(validation_batch))
assert float(tf.reduce_max(validation_batch)) <= 255
validation_probs = saved_model(validation_batch, training=False).numpy()
assert validation_probs.shape == (3, 6) and np.isfinite(validation_probs).all()
assert np.allclose(validation_probs.sum(axis=1), 1.0, atol=1e-4)

historical = pd.read_csv(HISTORICAL_VAL_PATH).set_index('relative_path', verify_integrity=True)
max_abs_diff = 0.0
for i, row in val_smoke.iterrows():
    old = historical.loc[row['relative_path']]
    if int(validation_probs[i].argmax()) != int(old['predicted_id']):
        raise RuntimeError('Validation predictions disagree with historical U1 results.')
    historical_probs = np.asarray([
        float(old[f'prob_{j}_{CLASS_NAMES[j].replace(" ", "_")}'])
        for j in range(6)
    ], dtype=np.float64)
    max_abs_diff = max(max_abs_diff, float(np.abs(validation_probs[i] - historical_probs).max()))
if max_abs_diff > 1e-3:
    raise RuntimeError(f'Unexpected difference from historical validation scores: {max_abs_diff}')

print('Frozen model:', saved_model.name)
print('Model input/output:', saved_model.input_shape, saved_model.output_shape)
print('Validation-only smoke predictions matched:', len(val_smoke), '/', len(val_smoke))
print('Maximum validation probability difference:', max_abs_diff)
print('SECTION 3 PASS: Original U1 preprocessing/inference reproduced. Test pixels untouched.')

Frozen model: Ultrasound_MobileNetV2_frozen_baseline
Model input/output: (None, 224, 224, 3) (None, 6)
Validation-only smoke predictions matched: 3 / 3
Maximum validation probability difference: 3.0232650760630975e-07
SECTION 3 PASS: Original U1 preprocessing/inference reproduced. Test pixels untouched.


## Section 4: Check test archive inventory and save the final protocol (metadata only)

Inspect **ZIP filenames and member sizes, not the test image contents**. Save a new test protocol under `ultrasound/selection/` with the exact model/manifest hashes, image counts, order, preprocessing, metrics and output path. The saved freeze JSON is **not modified**. This is the last routine preflight review before explicit authorization. If a matching protocol already exists (for example after a Colab reset), verify and reuse it read-only. If any TEST inference is already complete, skip Sections 5–6 and generate/verify the saved report only.

In [4]:
EXPECTED_ZIP_MD5 = '2a5fcc2cefb789bcc0f6c1f73e0ea43f'
RUN_NAME = 'U_Final_Test_U1_seed42'
if TEST_RUN_STATE not in ('NOT_STARTED', 'INFERENCE_COMPLETED', 'FINALIZED'):
    raise RuntimeError('Unknown TEST run state. Stop.')

with zipfile.ZipFile(ZIP_PATH, 'r') as archive:
    names = archive.namelist()
    duplicates = set()
    seen = set()
    for name in names:
        if name in seen:
            duplicates.add(name)
        seen.add(name)
    if duplicates.intersection(set(test_rows['relative_path'])):
        raise RuntimeError('Duplicate test member names in archive.')
    total_raw_bytes = 0
    for rel in test_rows['relative_path'].astype(str):
        member = validate_zip_member(rel, archive)
        total_raw_bytes += member.file_size

protocol = {
    'project': 'Explainable Transfer Learning Framework for Fetal Ultrasound Standard Plane Classification',
    'experiment': RUN_NAME,
    'protocol_recorded_utc': datetime.now(timezone.utc).isoformat(),
    'selection_freeze_relative_path': str(FREEZE_PATH.relative_to(ROOT)),
    'selection_freeze_sha256': sha256_file(FREEZE_PATH),
    'selected_model': 'U1_MobileNetV2_frozen_seed42',
    'checkpoint': {'relative_path': str(MODEL_PATH.relative_to(ROOT)),
                   'sha256': sha256_file(MODEL_PATH)},
    'configuration_sha256': sha256_file(CONFIG_PATH),
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'class_mapping_sha256': sha256_file(MAPPING_PATH),
    'audit_sha256': sha256_file(AUDIT_PATH),
    'dataset_archive': {'relative_path': str(ZIP_PATH.relative_to(ROOT)),
                        'expected_md5': EXPECTED_ZIP_MD5},
    'split': 'test',
    'held_out_role': 'original official FETAL_PLANES_DB test split',
    'split_unit': 'patient',
    'test_images': 5271,
    'test_patient_groups': 896,
    'test_order_sha256': TEST_ORDER_SHA,
    'class_mapping': class_to_id,
    'preprocessing': ('tf.io.decode_image(channels=3, expand_animations=False); '
                      'tf.image.resize_with_pad(224,224); float32 [0,255]; '
                      'model-internal Rescaling(1/127.5, -1); no augmentation'),
    'batch_size': BATCH_SIZE,
    'inference_mode': 'saved_model(x, training=False)',
    'metrics_predeclared': [
        'accuracy', 'macro_precision', 'macro_recall', 'macro_f1',
        'per_class_precision_recall_f1_support', 'confusion_matrix'
    ],
    'no_training': True,
    'no_test_based_tuning_or_model_reselection': True,
    'one_planned_test_evaluation': True,
    'results_relative_path': str(RESULT_DIR.relative_to(ROOT)),
    'status': 'PLANNED_NOT_EVALUATED',
    'notes': ('Decision was made after validation comparisons, before the test; '
              'not a preregistered research protocol. Do not treat TEST as VALIDATION.'),
}
PROTOCOL_PATH.parent.mkdir(parents=True, exist_ok=True)
if PROTOCOL_PATH.exists():
    existing_protocol = json.loads(PROTOCOL_PATH.read_text(encoding='utf-8'))
    for field, value in protocol.items():
        if field == 'protocol_recorded_utc':
            continue
        if existing_protocol.get(field) != value:
            raise RuntimeError('Existing test protocol differs: ' + field)
    protocol = existing_protocol
    print('Existing immutable TEST protocol verified; not overwritten.')
else:
    if TEST_RUN_STATE != 'NOT_STARTED':
        raise RuntimeError('Saved TEST inference exists without its protocol. Stop.')
    with PROTOCOL_PATH.open('x', encoding='utf-8') as fh:
        json.dump(protocol, fh, indent=2, ensure_ascii=False)
        fh.write('\n')
assert json.loads(PROTOCOL_PATH.read_text(encoding='utf-8')) == protocol

print('TEST archive inventory names matched:', len(test_rows))
print('TEST ZIP uncompressed member size:', round(total_raw_bytes / 1024**3, 3), 'GiB')
print('Protocol:', PROTOCOL_PATH.relative_to(ROOT))
print('Output directory reserved (not yet created):', RESULT_DIR.relative_to(ROOT))
print('Original official test images:', len(test_rows))
print('SECTION 4 PASS: Final protocol recorded before test. Test pixels not read.')

TEST archive inventory names matched: 5271
TEST ZIP uncompressed member size: 0.899 GiB
Protocol: ultrasound/selection/U1_final_test_protocol_2026-10-19.json
Output directory reserved (not yet created): ultrasound/results/U_Final_Test_U1_seed42
Original official test images: 5271
SECTION 4 PASS: Final protocol recorded before test. Test pixels not read.


## Section 5: Explicit final-test authorization (still no test pixels)

**Pause and review Section 4's output first.** This is a deliberately interactive gate. Type the exact displayed authorization phrase into the input prompt *only if you intend to perform the one planned final held-out evaluation*. This section does not yet decode test images. If authorization is not granted, the notebook stops without test access. The authorization does not permit tuning or repeat testing.

In [5]:
if not PROTOCOL_PATH.is_file():
    raise RuntimeError('Final protocol missing. Do not run the test.')
assert json.loads(PROTOCOL_PATH.read_text(encoding='utf-8')) == protocol
assert sha256_file(FREEZE_PATH) == protocol['selection_freeze_sha256']
assert sha256_file(MODEL_PATH) == protocol['checkpoint']['sha256']
assert sha256_file(MANIFEST_PATH) == protocol['manifest_sha256']
assert TEST_ORDER_SHA == protocol['test_order_sha256']

if RESULT_DIR.exists():
    if TEST_RUN_STATE not in ('INFERENCE_COMPLETED', 'FINALIZED'):
        raise FileExistsError('Existing TEST attempt; do not authorize a new inference.')
    print('SECTION 5 SKIPPED: Existing TEST inference found; never authorize another pass.')
    print('For report-only recovery, skip Section 6 and continue at Section 7 or 8.')
else:
    CONFIRM_PHRASE = 'AUTHORIZE U1 FINAL TEST ONCE'
    entered = input('To authorize exactly ONE frozen U1 held-out TEST evaluation, type '
                    + repr(CONFIRM_PHRASE) + ': ')
    if entered != CONFIRM_PHRASE:
        raise PermissionError('Final TEST not authorized. No test images accessed.')

    AUTHORIZATION = {
        'authorized_at_utc': datetime.now(timezone.utc).isoformat(),
        'exact_phrase_confirmed': True,
        'protocol_sha256': sha256_file(PROTOCOL_PATH),
        'model_sha256': sha256_file(MODEL_PATH),
        'test_order_sha256': TEST_ORDER_SHA,
    }
    print('SECTION 5 PASS: Explicit one-time TEST authorization recorded in runtime memory.')
    print('No test image pixels read yet. Next section begins the irreversible TEST evaluation.')

To authorize exactly ONE frozen U1 held-out TEST evaluation, type 'AUTHORIZE U1 FINAL TEST ONCE': AUTHORIZE U1 FINAL TEST ONCE
SECTION 5 PASS: Explicit one-time TEST authorization recorded in runtime memory.
No test image pixels read yet. Next section begins the irreversible TEST evaluation.


## Section 6: ONE final held-out TEST inference pass (test pixels accessed here)

**Run only once, after reviewing Section 5.** Copy the already-saved source ZIP to Colab's temporary disk if needed; verify its official MD5. The copy is a local cache, not a new download. A fresh, distinct Drive result directory and `test_evaluation_started.json` receipt are created **before the first test image is decoded**.

Each test image is read **once** from the source ZIP, checked against its immutable manifest `sha256`, decoded using original U1 preprocessing, and scored by the saved U1 model with `training=False`. No augmentation, thresholds, gradient calculation or fitting is performed. Predictions are appended to a private Drive CSV per batch, with no patient IDs or image filenames. This makes partial evidence recoverable after a runtime failure. When all 5,271 rows are saved, an inference-complete receipt is written.

**If this section fails:** stop; share the error and any saved partial-record counts. **Do not rerun blindly**, and do not change the model in response to test performance.

In [6]:
import csv
from tqdm.auto import tqdm

if 'AUTHORIZATION' not in globals() or not AUTHORIZATION.get('exact_phrase_confirmed'):
    raise PermissionError('Section 5 authorization is required before reading test pixels.')
if RESULT_DIR.exists():
    raise FileExistsError('One-time TEST attempt already exists; do not rerun Section 6.')
if AUTHORIZATION['protocol_sha256'] != sha256_file(PROTOCOL_PATH):
    raise RuntimeError('Protocol changed after authorization.')
if sha256_file(FREEZE_PATH) != protocol['selection_freeze_sha256']:
    raise RuntimeError('Freeze changed after authorization.')
if sha256_file(MODEL_PATH) != protocol['checkpoint']['sha256']:
    raise RuntimeError('Frozen checkpoint changed after authorization.')
if sha256_file(MANIFEST_PATH) != protocol['manifest_sha256']:
    raise RuntimeError('Frozen manifest changed after authorization.')
if len(test_rows) != 5271 or TEST_ORDER_SHA != protocol['test_order_sha256']:
    raise RuntimeError('Test order/size changed after authorization.')

# Only now stage the existing archive locally for reliable sequential evaluation.
LOCAL_ZIP = Path('/content/U1_final_test_FETAL_PLANES_ZENODO.zip')
if not LOCAL_ZIP.is_file() or LOCAL_ZIP.stat().st_size != ZIP_PATH.stat().st_size:
    if shutil.disk_usage('/content').free < ZIP_PATH.stat().st_size + 2 * 1024**3:
        raise RuntimeError('Insufficient temporary Colab space for verified ZIP copy.')
    temporary_copy = Path(str(LOCAL_ZIP) + '.partial')
    if temporary_copy.exists():
        raise FileExistsError('Interrupted local ZIP copy exists; inspect before retrying.')
    print('Copying existing Drive archive to temporary Colab disk (not downloading)...')
    shutil.copyfile(ZIP_PATH, temporary_copy)
    temporary_copy.replace(LOCAL_ZIP)

md5 = hashlib.md5()
with LOCAL_ZIP.open('rb') as fh:
    for chunk in iter(lambda: fh.read(4 * 1024 * 1024), b''):
        md5.update(chunk)
if md5.hexdigest() != EXPECTED_ZIP_MD5:
    raise RuntimeError('Source dataset ZIP MD5 mismatch. Stop before final inference.')

# This run directory is the permanent one-time guard. Never overwrite it.
RESULT_DIR.mkdir(parents=True, exist_ok=False)
attempt = {
    'status': 'TEST_INFERENCE_STARTED',
    'started_at_utc': datetime.now(timezone.utc).isoformat(),
    'authorization_utc': AUTHORIZATION['authorized_at_utc'],
    'protocol_sha256': sha256_file(PROTOCOL_PATH),
    'freeze_sha256': sha256_file(FREEZE_PATH),
    'checkpoint_sha256': sha256_file(MODEL_PATH),
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'test_order_sha256': TEST_ORDER_SHA,
    'test_images_expected': 5271,
    'archive_md5': md5.hexdigest(),
    'no_training': True,
    'test_based_tuning_allowed': False,
}
with (RESULT_DIR / 'test_evaluation_started.json').open('x', encoding='utf-8') as fh:
    json.dump(attempt, fh, indent=2)
    fh.write('\n')

RAW_PRED_PATH = RESULT_DIR / 'test_predictions.csv'
prob_cols = [f'prob_{idx}' for idx in range(6)]
fieldnames = ['test_row_index', 'image_sha256', 'true_id', 'predicted_id'] + prob_cols
with RAW_PRED_PATH.open('x', encoding='utf-8', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()

n_saved = 0
with zipfile.ZipFile(LOCAL_ZIP, 'r') as archive:
    for start in tqdm(range(0, len(test_rows), BATCH_SIZE), desc='Frozen U1: one TEST pass'):
        batch_rows = test_rows.iloc[start:start + BATCH_SIZE]
        images = []
        for row in batch_rows.itertuples(index=False):
            validate_zip_member(row.relative_path, archive)
            raw = archive.read(row.relative_path)
            if hashlib.sha256(raw).hexdigest() != row.sha256:
                raise RuntimeError('TEST image failed original manifest hash check. STOP.')
            images.append(decode_like_u1(raw))
        x = tf.stack(images, axis=0)
        assert tuple(x.shape[1:]) == (224, 224, 3)
        scores = saved_model(x, training=False).numpy()
        if scores.shape != (len(batch_rows), 6) or not np.isfinite(scores).all():
            raise RuntimeError('Invalid TEST inference scores. Stop; preserve partial results.')
        if not np.allclose(scores.sum(axis=1), 1.0, atol=1e-4):
            raise RuntimeError('TEST inference scores do not sum to one.')
        predicted = np.argmax(scores, axis=1)
        records = []
        for j, row in enumerate(batch_rows.itertuples(index=False)):
            entry = {
                'test_row_index': int(start + j),
                'image_sha256': row.sha256,
                'true_id': int(row.class_id),
                'predicted_id': int(predicted[j]),
            }
            entry.update({f'prob_{k}': float(scores[j, k]) for k in range(6)})
            records.append(entry)
        with RAW_PRED_PATH.open('a', encoding='utf-8', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writerows(records)
        n_saved += len(records)

if n_saved != 5271:
    raise RuntimeError(f'Incomplete TEST inference: only {n_saved} rows saved.')

inference_complete = {
    'status': 'TEST_INFERENCE_COMPLETED',
    'completed_at_utc': datetime.now(timezone.utc).isoformat(),
    'test_images_scored': n_saved,
    'predictions_sha256': sha256_file(RAW_PRED_PATH),
    'test_order_sha256': TEST_ORDER_SHA,
    'checkpoint_sha256': sha256_file(MODEL_PATH),
    'no_training': True,
    'no_test_based_tuning': True,
}
with (RESULT_DIR / 'test_inference_completed.json').open('x', encoding='utf-8') as fh:
    json.dump(inference_complete, fh, indent=2)
    fh.write('\n')
print('Saved raw TEST predictions:', RAW_PRED_PATH)
print('Scored TEST images:', n_saved)
print('TEST inference checkpoint:', RESULT_DIR / 'test_inference_completed.json')
print('SECTION 6 PASS: Exactly one frozen U1 held-out TEST inference completed and saved.')

Copying existing Drive archive to temporary Colab disk (not downloading)...


Frozen U1: one TEST pass:   0%|          | 0/330 [00:00<?, ?it/s]

Saved raw TEST predictions: /content/drive/MyDrive/ICTIU_Final_Projects/ultrasound/results/U_Final_Test_U1_seed42/test_predictions.csv
Scored TEST images: 5271
TEST inference checkpoint: /content/drive/MyDrive/ICTIU_Final_Projects/ultrasound/results/U_Final_Test_U1_seed42/test_inference_completed.json
SECTION 6 PASS: Exactly one frozen U1 held-out TEST inference completed and saved.


## Section 7: Compute and export final TEST metrics from SAVED predictions (no inference)

**No model invocation, no ZIP access, no test pixel decoding.** Read Section 6's already-saved probability table, cross-check row order and manifest hashes, then compute the **predeclared** metrics and confusion matrix. Save genuine results, a provenance record, a report-ready summary and artifact checksums to the new test result folder. This cannot change the model or produce another evaluation pass. Do not compare and retune U2 after seeing TEST performance.

In [7]:
import importlib.metadata
import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
)

COMPLETE_PATH = RESULT_DIR / 'test_inference_completed.json'
STARTED_PATH = RESULT_DIR / 'test_evaluation_started.json'
FINAL_MARKER = RESULT_DIR / 'FINALIZED.json'
if FINAL_MARKER.exists():
    raise FileExistsError('This TEST result is already finalized. Use Section 8 only.')
if not (COMPLETE_PATH.is_file() and STARTED_PATH.is_file()):
    raise RuntimeError('Section 6 has not completed. Never infer metrics from partial rows.')

completed = json.loads(COMPLETE_PATH.read_text(encoding='utf-8'))
started = json.loads(STARTED_PATH.read_text(encoding='utf-8'))
assert completed['status'] == 'TEST_INFERENCE_COMPLETED'
assert completed['test_images_scored'] == 5271
assert completed['checkpoint_sha256'] == sha256_file(MODEL_PATH)
assert started['protocol_sha256'] == sha256_file(PROTOCOL_PATH)
assert completed['test_order_sha256'] == TEST_ORDER_SHA
assert completed['predictions_sha256'] == sha256_file(RAW_PRED_PATH)

predictions = pd.read_csv(RAW_PRED_PATH, dtype={'image_sha256': str})
assert len(predictions) == 5271
assert list(predictions.columns) == fieldnames
assert predictions['test_row_index'].tolist() == list(range(5271))
assert predictions['image_sha256'].tolist() == test_rows['sha256'].tolist()
assert predictions['true_id'].tolist() == test_rows['class_id'].astype(int).tolist()
assert predictions['predicted_id'].between(0, 5).all()
assert np.isfinite(predictions[prob_cols].to_numpy()).all()
assert np.allclose(predictions[prob_cols].sum(axis=1).to_numpy(), 1.0, atol=1e-4)

y_true = predictions['true_id'].to_numpy(dtype=np.int32)
y_pred = predictions['predicted_id'].to_numpy(dtype=np.int32)
labels = list(range(6))
accuracy = float(accuracy_score(y_true, y_pred))
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, average='macro', zero_division=0
)
per_p, per_r, per_f1, per_support = precision_recall_fscore_support(
    y_true, y_pred, labels=labels, average=None, zero_division=0
)
cm = confusion_matrix(y_true, y_pred, labels=labels)
assert cm.shape == (6, 6) and int(cm.sum()) == 5271

metrics = {
    'partition': 'TEST',
    'experiment': RUN_NAME,
    'selected_model': 'U1_MobileNetV2_frozen_seed42',
    'n_images': 5271,
    'n_patient_groups': 896,
    'accuracy': accuracy,
    'macro_precision': float(macro_precision),
    'macro_recall': float(macro_recall),
    'macro_f1': float(macro_f1),
    'n_correct': int((y_true == y_pred).sum()),
    'n_errors': int((y_true != y_pred).sum()),
    'source_predictions_sha256': sha256_file(RAW_PRED_PATH),
    'no_post_test_tuning': True,
}
per_class = pd.DataFrame({
    'class_id': labels,
    'class_name': CLASS_NAMES,
    'precision': per_p.astype(float),
    'recall': per_r.astype(float),
    'f1': per_f1.astype(float),
    'support': per_support.astype(int),
})

STAGE = Path('/content/U_Final_Test_U1_report_staging')
if STAGE.exists():
    raise FileExistsError('Existing local report staging; inspect before rerunning: ' + str(STAGE))
STAGE.mkdir(parents=True)
(STAGE / 'test_metrics.json').write_text(json.dumps(metrics, indent=2) + '\n', encoding='utf-8')
per_class.to_csv(STAGE / 'test_per_class_metrics.csv', index=False)
pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES).to_csv(
    STAGE / 'test_confusion_matrix.csv', index_label='true_class'
)
report_text = classification_report(
    y_true, y_pred, labels=labels, target_names=CLASS_NAMES,
    digits=4, zero_division=0
)
(STAGE / 'test_classification_report.txt').write_text(report_text, encoding='utf-8')
report_dict = classification_report(
    y_true, y_pred, labels=labels, target_names=CLASS_NAMES,
    output_dict=True, zero_division=0
)
pd.DataFrame(report_dict).transpose().to_csv(STAGE / 'test_classification_report.csv')

fig, ax = plt.subplots(figsize=(10, 8))
ConfusionMatrixDisplay(cm, display_labels=CLASS_NAMES).plot(
    ax=ax, xticks_rotation=40, colorbar=False, values_format='d'
)
ax.set_title('Frozen U1 MobileNetV2: final held-out TEST')
fig.tight_layout()
fig.savefig(STAGE / 'test_confusion_matrix.png', dpi=200, bbox_inches='tight')
plt.close(fig)

environment = {
    'python': platform.python_version(),
    'tensorflow': tf.__version__,
    'packages': {},
    'visible_gpus': [str(dev) for dev in tf.config.list_physical_devices('GPU')],
}
for package in ['keras', 'numpy', 'pandas', 'scikit-learn', 'matplotlib']:
    try:
        environment['packages'][package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        environment['packages'][package] = 'not installed'
(STAGE / 'test_environment.json').write_text(
    json.dumps(environment, indent=2) + '\n', encoding='utf-8'
)

provenance = {
    'report_status': 'FINAL_HELD_OUT_TEST_EVALUATED_ONCE',
    'partition': 'TEST',
    'completed_at_utc': completed['completed_at_utc'],
    'report_generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'protocol_file': str(PROTOCOL_PATH.relative_to(ROOT)),
    'protocol_sha256': sha256_file(PROTOCOL_PATH),
    'freeze_file': str(FREEZE_PATH.relative_to(ROOT)),
    'freeze_sha256': sha256_file(FREEZE_PATH),
    'model_sha256': sha256_file(MODEL_PATH),
    'configuration_sha256': sha256_file(CONFIG_PATH),
    'manifest_sha256': sha256_file(MANIFEST_PATH),
    'mapping_sha256': sha256_file(MAPPING_PATH),
    'archive_md5': started['archive_md5'],
    'test_order_sha256': TEST_ORDER_SHA,
    'predictions_sha256': sha256_file(RAW_PRED_PATH),
    'test_images': 5271,
    'test_patient_groups': 896,
    'training_performed': False,
    'model_selection_or_tuning_on_test': False,
    'notes': ('Original official patient-disjoint held-out test. '
              'Only one frozen U1 inference pass. No external-site or clinical validation.'),
}
(STAGE / 'test_run_provenance.json').write_text(
    json.dumps(provenance, indent=2) + '\n', encoding='utf-8'
)

summary = (
    '# Frozen U1 MobileNetV2: Final Held-out TEST\n\n'
    '**Test evaluation:** One planned pass on 5,271 official held-out ultrasound images '
    'from 896 unseen patient groups.\n\n'
    f'- TEST accuracy: **{accuracy:.4%}**\n'
    f'- TEST macro-precision: **{macro_precision:.6f}**\n'
    f'- TEST macro-recall: **{macro_recall:.6f}**\n'
    f'- TEST macro-F1: **{macro_f1:.6f}**\n'
    f'- TEST correct: **{metrics["n_correct"]}/5271**\n'
    '- Model: frozen U1 MobileNetV2 selected based on earlier validation comparisons.\n'
    '- Original test images were not used to select, train, or tune the model.\n'
    '- Test and historical validation metrics must be presented separately.\n\n'
    'This is not an external-hospital study or a clinically validated diagnostic tool.\n'
    'Softmax scores should not be interpreted as calibrated clinical confidence.\n'
)
(STAGE / 'RUN_SUMMARY.md').write_text(summary, encoding='utf-8')

# Publish only new report files; original inference receipts/CSV remain untouched.
for path in sorted(STAGE.iterdir()):
    destination = RESULT_DIR / path.name
    if destination.exists():
        raise FileExistsError('Existing TEST report artifact must not be overwritten: ' + str(destination))
    shutil.copy2(path, destination)
    assert sha256_file(path) == sha256_file(destination)

with FINAL_MARKER.open('x', encoding='utf-8') as fh:
    json.dump({'status': 'TEST_REPORT_FINALIZED',
               'created_utc': datetime.now(timezone.utc).isoformat(),
               'test_images': 5271,
               'test_predictions_sha256': sha256_file(RAW_PRED_PATH)}, fh, indent=2)
    fh.write('\n')

checksums = {
    str(p.relative_to(RESULT_DIR)): sha256_file(p)
    for p in sorted(RESULT_DIR.iterdir())
    if p.is_file() and p.name != 'artifact_checksums.json'
}
with (RESULT_DIR / 'artifact_checksums.json').open('x', encoding='utf-8') as fh:
    json.dump(checksums, fh, indent=2)
    fh.write('\n')

print('FINAL HELD-OUT TEST: Ultrasound U1 (5,271 images)')
print('TEST accuracy:', f'{accuracy:.4%}')
print('TEST macro-F1:', f'{macro_f1:.6f}')
print('TEST macro-precision:', f'{macro_precision:.6f}')
print('TEST macro-recall:', f'{macro_recall:.6f}')
print('TEST confusion matrix saved, rows=true / cols=predicted.')
print('Saved final TEST report:', RESULT_DIR)
print('SECTION 7 PASS: Final TEST metrics exported from existing predictions; no repeat inference.')

FINAL HELD-OUT TEST: Ultrasound U1 (5,271 images)
TEST accuracy: 85.4297%
TEST macro-F1: 0.834528
TEST macro-precision: 0.818983
TEST macro-recall: 0.860314
TEST confusion matrix saved, rows=true / cols=predicted.
Saved final TEST report: /content/drive/MyDrive/ICTIU_Final_Projects/ultrasound/results/U_Final_Test_U1_seed42
SECTION 7 PASS: Final TEST metrics exported from existing predictions; no repeat inference.


## Section 8: Read-only final integrity and provenance verification

Verify the complete saved artifacts, source checkpoint, class order, frozen split hashes, prediction count, inference receipt and report metadata. This section neither decodes test images nor recomputes test predictions. Save the **executed notebook** and keep the private Drive evidence; avoid publishing sample hashes, patient metadata or raw archives to a public GitHub repository by default.

In [8]:
if not (RESULT_DIR / 'FINALIZED.json').is_file():
    raise RuntimeError('Final TEST report is not complete. Stop and inspect earlier sections.')
checksums = json.loads((RESULT_DIR / 'artifact_checksums.json').read_text(encoding='utf-8'))
for rel, digest in checksums.items():
    if sha256_file(RESULT_DIR / rel) != digest:
        raise RuntimeError('Saved TEST artifact checksum changed: ' + rel)

saved_metrics = json.loads((RESULT_DIR / 'test_metrics.json').read_text(encoding='utf-8'))
saved_provenance = json.loads((RESULT_DIR / 'test_run_provenance.json').read_text(encoding='utf-8'))
saved_final = json.loads((RESULT_DIR / 'FINALIZED.json').read_text(encoding='utf-8'))
assert saved_metrics['partition'] == 'TEST'
assert saved_metrics['n_images'] == saved_final['test_images'] == 5271
assert saved_metrics['n_patient_groups'] == 896
assert saved_provenance['report_status'] == 'FINAL_HELD_OUT_TEST_EVALUATED_ONCE'
assert saved_provenance['training_performed'] is False
assert saved_provenance['model_selection_or_tuning_on_test'] is False
assert sha256_file(MODEL_PATH) == freeze['model_artifact']['sha256']
assert sha256_file(MANIFEST_PATH) == freeze['manifest']['sha256']
assert sha256_file(MAPPING_PATH) == freeze['mapping_file']['sha256']
assert sha256_file(FREEZE_PATH) == saved_provenance['freeze_sha256']
assert sha256_file(RESULT_DIR / 'test_predictions.csv') == saved_metrics['source_predictions_sha256']

print('Verified saved TEST artifacts:', len(checksums))
print('Frozen U1 checkpoint unchanged:', True)
print('Patient-disjoint official TEST images:', saved_metrics['n_images'])
print('TEST accuracy:', f"{saved_metrics['accuracy']:.4%}")
print('TEST macro-F1:', f"{saved_metrics['macro_f1']:.6f}")
print('Retuning based on TEST results: NOT PERMITTED')
print('SECTION 8 PASS: Ultrasound U1 final held-out TEST report and provenance verified.')

Verified saved TEST artifacts: 13
Frozen U1 checkpoint unchanged: True
Patient-disjoint official TEST images: 5271
TEST accuracy: 85.4297%
TEST macro-F1: 0.834528
Retuning based on TEST results: NOT PERMITTED
SECTION 8 PASS: Ultrasound U1 final held-out TEST report and provenance verified.


## Reporting and preservation notes

- Report **U1/U2 validation comparisons** separately from the **one final held-out TEST** of the preselected U1 checkpoint. Do not call historical 88.10% / 0.8572 TEST scores.
- The original official test contained 5,271 ultrasound images from 896 patient groups. Even a patient-separated held-out test from the *same dataset* is **not external-hospital validation**.
- Include per-class errors, especially the historical `Other` and fetal-abdomen limitations. Explain Grad-CAM as illustrative attribution on **validation images only**.
- Do not use this TEST result to add augmentation, tune thresholds, choose U2, or run the final test a second time. Any genuine procedural defect must be reported and investigated transparently before interpreting conclusions.
- Export the authentic completed notebook for the standalone Ultrasound repository. Do not alter the historical original setup, manifest, selected checkpoint, or freeze record.